In [0]:
from pyspark.sql import SparkSession

import mlflow
import mlflow.spark

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression ,RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

from pyspark.sql.functions import col
from pyspark.ml.functions import vector_to_array

from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType, ArrayType
from pyspark.ml.linalg import VectorUDT

spark = SparkSession.builder.appName("modelTraining").getOrCreate()

In [0]:
feature_df = spark.table("teleco_churn_datalakehouse.gold.customer_churn_features")

feature_df.display()

In [0]:
# Test - remove nulls from feature_df
feature_df = feature_df.dropna(subset=[
    "Tenure_Months",
    "Monthly_Charges",
    "Total_Charges"
])

- Creating the vector feature column using vectorAssembler

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "Tenure_Months",
        "Monthly_Charges",
        "Total_Charges"
    ],
    outputCol="features"
)

df_features = assembler.transform(feature_df)

- Creating the Train/test split in pyspark

In [0]:
train_df, test_df = df_features.randomSplit([0.8, 0.2], seed=42)

- veryfying the split

In [0]:
print("Training rows:", train_df.count())
print("Test rows:", test_df.count())

- Train the model with logistic regression.

In [0]:
mlflow.end_run()
with mlflow.start_run(run_name="Logistic_Regression_Model"):


    lr = LogisticRegression(
        featuresCol="features",
        labelCol="churn_label"
    )

    lr_model = lr.fit(train_df)

    lr_predictions = lr_model.transform(test_df)

    evaluator = BinaryClassificationEvaluator(
        labelCol="churn_label"
    )

    lr_auc = evaluator.evaluate(lr_predictions)

    print("Logistic Regression AUC:", lr_auc)

- Making predictions

In [0]:
predictions = lr_model.transform(test_df)

- Trainign the model using random forest classifier

- Evaluating the model

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="churn_label"
)

auc = evaluator.evaluate(predictions)

print("AUC:", auc)

In [0]:
# random forest classifier
mlflow.end_run()
with mlflow.start_run(run_name="Random_Forest_Model"):

    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol="churn_label",
        numTrees=100
    )

    rf_model = rf.fit(train_df)

    rf_predictions = rf_model.transform(test_df)

    evaluator = BinaryClassificationEvaluator(
        labelCol="churn_label"
    )

    rf_auc = evaluator.evaluate(rf_predictions)

    print("Random Forest AUC:", rf_auc)

In [0]:
# MLflow requires a UC volume path for dfs_tmpdir -  A path for storing the trained models
temp_path = "/Volumes/teleco_churn_datalakehouse/telecom_ml_database/models/tmp"

In [0]:
mlflow.spark.log_model(
    rf_model,
    "best_churn_model",
    dfs_tmpdir=temp_path
)

- Create a final predictions dataset

In [0]:
best_predictions = rf_model.transform(df_features)

In [0]:
%sql
-- Creating a volume for saving my model
CREATE VOLUME IF NOT EXISTS teleco_churn_datalakehouse.telecom_ml_database.models;

In [0]:
# Saving the model
# The path format is: /Volumes/<catalog>/<schema>/<volume>/<path>
lr_model.write().overwrite().save("/Volumes/teleco_churn_datalakehouse/telecom_ml_database/models/logistic_regression_model")

rf_model.write().overwrite().save("/Volumes/teleco_churn_datalakehouse/telecom_ml_database/models/random_forest_model")

- Creating the final predictions dataset

In [0]:
best_predictions = rf_model.transform(df_features)

- Extract churn probability

In [0]:
best_predictions = best_predictions.withColumn(
    "churn_probability",
    vector_to_array("probability")[1]
)

- Save the predictions table

In [0]:
# Extract the second element (probability of class 1)
extract_probability = udf(lambda v: float(v[1]) if v is not None else None, DoubleType())

# For binary classification, you can also use vectorToArray to convert to array first
from pyspark.ml.functions import vector_to_array

best_predictions = best_predictions.withColumn(
    "churn_probability", 
    vector_to_array("probability")[1]  # Direct array access after conversion
)

# Then select and save
best_predictions.select(
    "customer_key",
    "Tenure_Months", 
    "Monthly_Charges",
    "Total_Charges",
    "churn_probability",
    "prediction"
).write.format("delta").mode("overwrite").saveAsTable(
    "teleco_churn_datalakehouse.telecom_ml_database.churn_predictions_final"
)

- Create risk segments

In [0]:
from pyspark.sql.functions import when

risk_segments = best_predictions.withColumn(
    "risk_segment",
    when(col("churn_probability") > 0.7, "High Risk")
    .when(col("churn_probability") > 0.4, "Medium Risk")
    .otherwise("Low Risk")
)

In [0]:
risk_segments.write.format("delta").mode("overwrite").saveAsTable(
    "teleco_churn_datalakehouse.telecom_ml_database.customer_risk_segments"
)

- Confrming data into the churn prediction final

In [0]:
%sql
SELECT * 
FROM teleco_churn_datalakehouse.telecom_ml_database.churn_predictions_final
LIMIT 1000 ;

In [0]:
%sql
SELECT * 
FROM teleco_churn_datalakehouse.telecom_ml_database.customer_risk_segments
LIMIT 1000 ;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS teleco_churn_datalakehouse.telecom_ml_database ;

In [0]:
# will look into whether the training data should be saved as a table

train_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("teleco_churn_datalakehouse.telecom_ml_database.train_table")

test_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("teleco_churn_datalakehouse.telecom_ml_database.test_table")